In [ ]:
"""Train the encoder, decoder, discriminator, and MBTI classifier models."""

In [ ]:
from torch import nn, optim
import torch

In [ ]:
from caae import Classifier, Decoder, Discriminator, Encoder
from constants import BATCH_SIZE, LABEL_DIM, LATENT_DIM, NUM_CLASSES
from constants import LATENT_DIM, NUM_EPOCHS, EMBEDDING_DIM
from dataset import MBTIDataset, DATASET_SIZE
from torch.utils.data import DataLoader
from embeddings import generate_embeddings

In [ ]:
# The transformer embeddings determine the network input width.
input_dim = EMBEDDING_DIM

In [ ]:
mbti_dataset = MBTIDataset(size=DATASET_SIZE)
sentence_embeddings,label_onehots = generate_embeddings(mbti_dataset)
mbti_dataset.sentence_embeddings = sentence_embeddings
mbti_dataset.label_one_hots = label_onehots

In [ ]:
encoder = Encoder(input_dim, LATENT_DIM)
decoder = Decoder(LATENT_DIM, LABEL_DIM, input_dim)
discriminator = Discriminator(LATENT_DIM)
classifier = Classifier(LATENT_DIM, num_classes=NUM_CLASSES)

In [ ]:
loader = DataLoader(mbti_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
optim_enc = optim.Adam(encoder.parameters(), lr=1e-3)
optim_dec = optim.Adam(decoder.parameters(), lr=1e-3)
optim_disc = optim.Adam(discriminator.parameters(), lr=1e-3)
optim_cls = optim.Adam(classifier.parameters(), lr=1e-3)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
bce = nn.BCELoss()
mse = nn.MSELoss()

In [ ]:
for epoch in range(NUM_EPOCHS):
    for x, labels in loader:
        # Encode
        z = encoder(x)

        # Teach the discriminator to distinguish sampled and encoded latents.
        real = torch.ones(x.size(0),1)
        fake = torch.zeros(x.size(0),1)
        disc_real = discriminator(torch.randn_like(z))
        disc_fake = discriminator(z.detach())
        loss_disc = bce(disc_real, real) + bce(disc_fake, fake)
        optim_disc.zero_grad(); loss_disc.backward(); optim_disc.step()

        # Encourage the encoder to produce latents the discriminator accepts.
        disc_fake = discriminator(z)
        loss_enc_adv = bce(disc_fake, real)

        # Reconstruct the original embedding using the latent vector and label.
        x_hat = decoder(z, labels)
        loss_rec = mse(x_hat, x)

        # Predict the MBTI class represented by the one-hot target label.
        logits = classifier(z)
        target = torch.argmax(labels, dim=1)
        loss_cls = loss_fn(logits, target)

        # Optimize the reconstruction, adversarial, and classification objectives.
        loss_total = loss_rec + loss_enc_adv + loss_cls
        optim_enc.zero_grad(); optim_dec.zero_grad(); optim_cls.zero_grad()
        loss_total.backward()
        optim_enc.step(); optim_dec.step(); optim_cls.step()

    print(f"📈 Epoch {epoch + 1:02d}/{NUM_EPOCHS} complete  |  loss: {loss_total.item():.4f}")